In [1]:
import warnings, json, time, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

from sklift.metrics import uplift_at_k, qini_auc_score
from sklift.metrics import uplift_curve, qini_curve

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Constants ──────────────────────────────────────────────────────────────────
RANDOM_STATE   = 42
N_FOLDS        = 5
ARTIFACTS_DIR  = 'pilot_artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TARGET_COL    = 'rec_spend'
TREATMENT_COL = 'treatment_flg'
ID_COL        = 'user_id'
COMM_COL      = 'communication_type'

COMM_TYPES    = ['com_type_1', 'com_type_2', 'com_type_3']

# TOP-25 из EDA feature importance
TOP25 = [
    'cus_cat_7_rto', 'n_redeem', 'cus_cat_7_atv', 'cus_cat_5_rto',
    'cus_cat_5_std', 'mntv', 'n_days_life', 'cus_cat_5_atv',
    'cus_cat_7_n_days_big_period', 'cus_cat_7_std', 'cus_mark_n_offers',
    'cus_cat_6_rto', 'rto_format_1', 'age', 'cus_mark_n_view', 'n_sku',
    'cus_cat_6_std', 'cus_cat_6_atv', 'cus_cat_5_max_min_days_diff',
    'cus_mark_n_rule', 'p_25_tv', 'cus_cat_7_max_min_days_diff',
    'mxtv', 'rto_format_3', 'avg_days_btw_trn'
]

print('✅ Config loaded')
print(f'Artifacts dir: {ARTIFACTS_DIR}/')

✅ Config loaded
Artifacts dir: pilot_artifacts/


In [2]:
train = pd.read_parquet('train.parquet')
test  = pd.read_parquet('test.parquet')
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Treatment balance: T={train[TREATMENT_COL].mean():.3f}')

ALL_FEATS = [c for c in train.columns
             if c not in [ID_COL, TARGET_COL, TREATMENT_COL]]

# Label-encode строковые колонки
le_map = {}
for col in train[ALL_FEATS].select_dtypes(include=['object','string','category']).columns:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], ignore_index=True).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_map[col] = le

print(f'Features: {len(ALL_FEATS)}')
print(f'Target zeros: {(train[TARGET_COL]==0).mean():.1%}')

Train: (355246, 89), Test: (118414, 87)
Treatment balance: T=0.496
Features: 86
Target zeros: 90.2%


In [3]:
def engineer_features(df):
    df = df.copy()
    eps = 1e-6

    # Affinity to target category
    df['cat7_share']       = df['cus_cat_7_rto'] / (df['rto'] + eps)
    df['cat6_share']       = df['cus_cat_6_rto'] / (df['rto'] + eps)
    df['cat5_share']       = df['cus_cat_5_rto'] / (df['rto'] + eps)

    # Marketing responsiveness
    df['mkt_resp_rate']    = df['cus_mark_n_rule'] / (df['cus_mark_n_offers'].clip(lower=1))
    df['mkt_view_rate']    = df['cus_mark_n_view']  / (df['cus_mark_n_offers'].clip(lower=1))

    # Recency relative to avg purchase cadence
    df['recency_ratio']    = df['cus_cat_7_last_1_days'] / (df['avg_days_btw_trn'] + eps)

    # Spend variability
    df['spend_cv']         = df['stdtv'] / (df['atv'] + eps)

    # Days life × transaction frequency
    df['trn_density']      = df['n_trn'] / (df['n_days_life'] + eps) * 365

    # Category breadth in target hierarchy
    df['cat_breadth_ratio'] = df['n_cat_7'] / (df['n_cat_5'] + eps)

    # How fresh is the category purchase vs general last visit
    df['cat7_vs_last_visit'] = df['cus_cat_7_last_1_days'] / (df['n_days_last_visit'] + eps)

    return df

train = engineer_features(train)
test  = engineer_features(test)

NEW_FEATS = [
    'cat7_share','cat6_share','cat5_share',
    'mkt_resp_rate','mkt_view_rate',
    'recency_ratio','spend_cv','trn_density',
    'cat_breadth_ratio','cat7_vs_last_visit'
]

FEAT_COLS = ALL_FEATS + NEW_FEATS
FEAT_COLS_TOP = TOP25 + NEW_FEATS  # расширенный топ

print(f'Total features after engineering: {len(FEAT_COLS)}')
print(f'New features: {NEW_FEATS}')

Total features after engineering: 96
New features: ['cat7_share', 'cat6_share', 'cat5_share', 'mkt_resp_rate', 'mkt_view_rate', 'recency_ratio', 'spend_cv', 'trn_density', 'cat_breadth_ratio', 'cat7_vs_last_visit']


In [4]:
def make_stratify_col(df):
    """6-страт: treatment × communication_type"""
    return df[TREATMENT_COL].astype(str) + '_' + df[COMM_COL].astype(str)

def uplift_at_k_spend(y_spend, uplift_scores, treatment, k=0.1):
    """
    uplift@k по непрерывному rec_spend:
    E[rec_spend | T=1, top-k%] - E[rec_spend | T=0, top-k%]
    """
    n = len(y_spend)
    top_n = max(int(n * k), 1)
    
    # Берём индексы топ-k% по uplift score
    top_idx = np.argsort(uplift_scores)[::-1][:top_n]
    
    t_mask  = treatment[top_idx] == 1
    c_mask  = treatment[top_idx] == 0
    
    if t_mask.sum() == 0 or c_mask.sum() == 0:
        return 0.0
    
    return float(y_spend[top_idx][t_mask].mean() - y_spend[top_idx][c_mask].mean())


def oof_uplift_at_k(y_true, scores, treatment, k=0.1, n_boot=100, seed=42):
    """Возвращает (mean, ci80_lo, ci80_hi) по rec_spend"""
    point = uplift_at_k_spend(y_true, scores, treatment, k)
    rng   = np.random.default_rng(seed)
    idx   = np.arange(len(y_true))
    boots = []
    for _ in range(n_boot):
        s = rng.choice(idx, len(idx), replace=True)
        boots.append(uplift_at_k_spend(y_true[s], scores[s], treatment[s], k))
    boots = np.array(boots)
    return float(point), float(np.percentile(boots, 10)), float(np.percentile(boots, 90))

def oof_qini(y_true, scores, treatment):
    y_bin = (y_true > 0).astype(int)
    return float(qini_auc_score(y_bin, scores, treatment))

def get_folds(df, n_splits=N_FOLDS):
    strat = make_stratify_col(df)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return list(skf.split(df, strat))

RESULTS = {}  # dict: model_name -> metrics

def register_result(name, y_val, scores_val, treat_val, elapsed):
    u10, ci_lo, ci_hi = oof_uplift_at_k(y_val, scores_val, treat_val)
    qauc = oof_qini(y_val, scores_val, treat_val)
    
    entry = {
        'model':       name,
        'uplift@10':   u10,
        'ci80_lo':     ci_lo,
        'ci80_hi':     ci_hi,
        'qini_auc':    qauc,
        'elapsed_sec': round(elapsed, 1)
    }
    
    # Обновляем если уже есть, иначе добавляем
    for i, r in enumerate(RESULTS):
        if r['model'] == name:
            RESULTS[i] = entry
            break
    else:
        RESULTS.append(entry)
    
    print(f'  uplift@10={u10:.4f}  CI80=[{ci_lo:.4f}, {ci_hi:.4f}]  '
          f'qini={qauc:.4f}  ({elapsed:.0f}s)')
    return u10

print('✅ Validation utils ready')

✅ Validation utils ready


In [6]:
BASE_LGB_REG = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=127,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
BASE_LGB_CLF = {**BASE_LGB_REG}

def encode_X(df, feat_cols):
    X = df[feat_cols].copy().fillna(-999)
    for c in X.select_dtypes(include=['object','string','category']).columns:
        X[c] = X[c].astype('category')
    return X

print('✅ LGB params ready')

✅ LGB params ready


In [5]:
print('Training Model 1: T-Learner...')
t0 = time.time()

folds = get_folds(train)
oof_scores_tl = np.zeros(len(train))

y  = train[TARGET_COL].values
T  = train[TREATMENT_COL].values

Training Model 1: T-Learner...


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# X-LEARNER OOF (единая функция для глобальной и per-channel модели)
# ═══════════════════════════════════════════════════════════════════════════════

def xlearner_oof(params1, params2, prop_model, folds_list,
                 df_sub, y_sub, T_sub, feat_cols):
    """
    X-Learner OOF:
      Stage1 : mu1 = f(X|T=1), mu0 = f(X|T=0)
      Stage2 : D1 = y1 - mu0(x1), D0 = mu1(x0) - y0  → tau1, tau0
      Final  : CATE = g(x)*tau0(x) + (1-g(x))*tau1(x)
    """
    oof = np.zeros(len(df_sub))

    for tr_idx, val_idx in folds_list:
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]

        mask1 = Ttr == 1
        mask0 = Ttr == 0

        mu1 = lgb.LGBMRegressor(**params1)
        mu0 = lgb.LGBMRegressor(**params1)
        mu1.fit(X_tr[mask1], ytr[mask1])
        mu0.fit(X_tr[mask0], ytr[mask0])

        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]

        tau1 = lgb.LGBMRegressor(**params2)
        tau0 = lgb.LGBMRegressor(**params2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        g.fit(X_tr, Ttr)
        g_val = g.predict_proba(X_val)[:, 1]

        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    return uplift_at_k_spend(y_sub, oof, T_sub, k=0.1), oof


# ═══════════════════════════════════════════════════════════════════════════════
# ФУНКЦИЯ: обучить X-Learner на всём трейне → предикт на тесте
# ═══════════════════════════════════════════════════════════════════════════════

def xlearner_predict_test(params1, params2, prop_model,
                          df_train, y_tr, T_tr,
                          df_test, feat_cols):
    X_tr = encode_X(df_train, feat_cols)
    X_te = encode_X(df_test,  feat_cols)

    mask1 = T_tr == 1
    mask0 = T_tr == 0

    mu1 = lgb.LGBMRegressor(**params1)
    mu0 = lgb.LGBMRegressor(**params1)
    mu1.fit(X_tr[mask1], y_tr[mask1])
    mu0.fit(X_tr[mask0], y_tr[mask0])

    D1 = y_tr[mask1] - mu0.predict(X_tr[mask1])
    D0 = mu1.predict(X_tr[mask0]) - y_tr[mask0]

    tau1 = lgb.LGBMRegressor(**params2)
    tau0 = lgb.LGBMRegressor(**params2)
    tau1.fit(X_tr[mask1], D1)
    tau0.fit(X_tr[mask0], D0)

    if prop_model == 'logreg':
        g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
    else:
        g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    g.fit(X_tr, T_tr)
    g_te = g.predict_proba(X_te)[:, 1]

    return g_te * tau0.predict(X_te) + (1 - g_te) * tau1.predict(X_te)

print('✅ xlearner_predict_test defined')


print('✅ xlearner_oof и xlearner_predict_test определены')

✅ xlearner_predict_test defined
✅ xlearner_oof и xlearner_predict_test определены


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# ГЛОБАЛЬНАЯ МОДЕЛЬ — 5 СИДОВ, ПОЛНЫЙ ТРЕЙН
# ═══════════════════════════════════════════════════════════════════════════════

print('=' * 60)
print('Global X-Learner: train on full data, 5 seeds')
print('=' * 60)

with open(f'{ARTIFACTS_DIR}/optuna_xl_params.json', 'r') as f:
    xl_saved = json.load(f)

best_params1_xl = xl_saved['params1']
best_params2_xl = xl_saved['params2']
best_prop_xl    = xl_saved['prop_model']
best_feat_xl    = FEAT_COLS if xl_saved['feat_set'] == 'all' else FEAT_COLS_TOP

print(f"prop={best_prop_xl}  feat_set={xl_saved['feat_set']}")
print(f"s1 n_estimators={best_params1_xl['n_estimators']}  "
      f"s2 n_estimators={best_params2_xl['n_estimators']}")

SEEDS = [42, 123, 777, 2024, 31415]

test_scores_global_seeds = np.zeros((len(SEEDS), len(test)))

for i, seed in enumerate(SEEDS):
    t0 = time.time()
    p1 = {**best_params1_xl, 'random_state': seed}
    p2 = {**best_params2_xl, 'random_state': seed}

    test_scores_global_seeds[i] = xlearner_predict_test(
        p1, p2, best_prop_xl,
        train, y, T,
        test, best_feat_xl
    )
    print(f'  seed={seed}  done ({time.time()-t0:.0f}s)')

test_global_avg = test_scores_global_seeds.mean(axis=0)
print(f'\nGlobal avg: min={test_global_avg.min():.3f}  '
      f'max={test_global_avg.max():.3f}  '
      f'mean={test_global_avg.mean():.3f}')

np.save(f'{ARTIFACTS_DIR}/test_global_xl_5seed.npy', test_global_avg)

submission_global = pd.DataFrame({
    'user_id':      test[ID_COL],
    'UPLIFT_SCORE': test_global_avg
})
submission_global.to_csv(f'{ARTIFACTS_DIR}/submission_xl_opt_5seed.csv', index=False)
print(f'✅ Saved: {ARTIFACTS_DIR}/submission_xl_opt_5seed.csv')

Global X-Learner: train on full data, 5 seeds
prop=logreg  feat_set=all
s1 n_estimators=815  s2 n_estimators=125


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  seed=42  done (81s)


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  seed=123  done (84s)


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  seed=777  done (82s)
  seed=2024  done (81s)
  seed=31415  done (80s)

Global avg: min=-49.854  max=401.009  mean=3.310
✅ Saved: pilot_artifacts/submission_xl_opt_5seed.csv


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PER-CHANNEL МОДЕЛЬ — 5 СИДОВ, ПОЛНЫЙ ТРЕЙН
# ═══════════════════════════════════════════════════════════════════════════════

print('=' * 60)
print('Per-Channel X-Learner: train on full data, 5 seeds')
print('=' * 60)

with open(f'{ARTIFACTS_DIR}/perchan_xl_params.json', 'r') as f:
    perchan_saved = json.load(f)

COMM_CODES = sorted(train[COMM_COL].unique())
print(f'Channels: {COMM_CODES}')

test_scores_perchan_seeds = np.zeros((len(SEEDS), len(test)))

for i, seed in enumerate(SEEDS):
    t0 = time.time()
    test_seed = np.zeros(len(test))

    for ch in COMM_CODES:
        ch_key = str(ch)
        ch_saved = perchan_saved[ch_key]

        p1_ch   = {**ch_saved['params1'], 'random_state': seed}
        p2_ch   = {**ch_saved['params2'], 'random_state': seed}
        prop_ch = ch_saved['prop']
        feat_ch = FEAT_COLS if ch_saved.get('feat_set', 'all') == 'all' else FEAT_COLS_TOP

        ch_mask_train = (train[COMM_COL] == ch).values
        ch_mask_test  = (test[COMM_COL]  == ch).values

        df_ch      = train[ch_mask_train].reset_index(drop=True)
        y_ch       = y[ch_mask_train]
        T_ch       = T[ch_mask_train]
        df_ch_test = test[ch_mask_test].reset_index(drop=True)

        test_seed[ch_mask_test] = xlearner_predict_test(
            p1_ch, p2_ch, prop_ch,
            df_ch, y_ch, T_ch,
            df_ch_test, feat_ch
        )

    test_scores_perchan_seeds[i] = test_seed
    print(f'  seed={seed}  done ({time.time()-t0:.0f}s)')

test_perchan_avg = test_scores_perchan_seeds.mean(axis=0)
print(f'\nPer-Channel avg: min={test_perchan_avg.min():.3f}  '
      f'max={test_perchan_avg.max():.3f}  '
      f'mean={test_perchan_avg.mean():.3f}')

np.save(f'{ARTIFACTS_DIR}/test_perchan_xl_5seed.npy', test_perchan_avg)

submission_perchan = pd.DataFrame({
    'user_id':      test[ID_COL],
    'UPLIFT_SCORE': test_perchan_avg
})
submission_perchan.to_csv(f'{ARTIFACTS_DIR}/submission_perchan_xl_opt_5seed.csv', index=False)
print(f'✅ Saved: {ARTIFACTS_DIR}/submission_perchan_xl_opt_5seed.csv')

Per-Channel X-Learner: train on full data, 5 seeds
Channels: [np.int64(0), np.int64(1), np.int64(2)]


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  seed=42  done (159s)


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ФИНАЛЬНЫЙ БЛЕНД: perchan × 0.7 + global × 0.3 (по рангу)
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.stats import rankdata

assert (submission_perchan['user_id'].values ==
        submission_global['user_id'].values).all(), 'user_id не совпадают!'

blend_test = 0.7 * rankdata(test_perchan_avg) + 0.3 * rankdata(test_global_avg)

submission_blend = pd.DataFrame({
    'user_id':      test[ID_COL],
    'UPLIFT_SCORE': blend_test
})

sub_path_blend = f'{ARTIFACTS_DIR}/submission_blend_perchan07_global03_5seed.csv'
submission_blend.to_csv(sub_path_blend, index=False)
print(f'✅ Saved: {sub_path_blend}')
print(f'   n={len(submission_blend):,}')
print(f'   stats: min={blend_test.min():.1f}  '
      f'max={blend_test.max():.1f}  '
      f'mean={blend_test.mean():.1f}')

# Состав топ-10% по каналам
n_top    = int(0.1 * len(test))
top_mask = blend_test >= np.sort(blend_test)[::-1][n_top]
print(f'\nСостав топ-10% теста ({n_top:,} клиентов):')
for ch in sorted(test[COMM_COL].unique()):
    n_ch = ((test[COMM_COL] == ch) & top_mask).sum()
    print(f'  ch={ch}: {n_ch:,} ({n_ch/n_top*100:.1f}%)')

In [ ]:
# ─── MODEL 9: Optuna-tuned X-Learner ─────────────────────────────────────────
print('Training Model 9: Optuna X-Learner...')
t0 = time.time()

# 40% subsample для поиска
from sklearn.model_selection import StratifiedShuffleSplit
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.6, random_state=RANDOM_STATE+1)
opt2_idx, _ = next(sss2.split(train, make_stratify_col(train)))
train_opt2  = train.iloc[opt2_idx].reset_index(drop=True)
y_opt2      = train_opt2[TARGET_COL].values
T_opt2      = train_opt2[TREATMENT_COL].values
folds_opt2  = list(StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
                   .split(train_opt2, make_stratify_col(train_opt2)))


def xlearner_oof(params1, params2, prop_model, folds_list, df_sub, y_sub, T_sub, feat_cols):
    """
    X-Learner OOF:
      Stage1: mu1 = f(X|T=1), mu0 = f(X|T=0)
      Stage2: D1 = y1 - mu0(x1), D0 = mu1(x0) - y0  →  tau1, tau0
      Final:  CATE = g(x)*tau0(x) + (1-g(x))*tau1(x)
    """
    from sklearn.linear_model import LogisticRegression
    oof = np.zeros(len(df_sub))

    for _, (tr_idx, val_idx) in enumerate(folds_list):
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]

        mask1 = Ttr == 1; mask0 = Ttr == 0

        # Stage 1
        mu1 = lgb.LGBMRegressor(**params1)
        mu0 = lgb.LGBMRegressor(**params1)
        mu1.fit(X_tr[mask1], ytr[mask1])
        mu0.fit(X_tr[mask0], ytr[mask0])

        # Imputed treatment effects
        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])  # treated: actual - mu0
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]  # control: mu1 - actual

        # Stage 2
        tau1 = lgb.LGBMRegressor(**params2)
        tau0 = lgb.LGBMRegressor(**params2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        # Propensity
        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
            g.fit(X_tr, Ttr)
            g_val = g.predict_proba(X_val)[:, 1]
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
            g.fit(X_tr, Ttr)
            g_val = g.predict_proba(X_val)[:, 1]

        # CATE = g*tau0 + (1-g)*tau1
        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    return uplift_at_k_spend(y_sub, oof, T_sub, k=0.1), oof


def objective_xl(trial):
    # Stage 1 params (outcome models — более сложные)
    params1 = dict(
        n_estimators      = trial.suggest_int('s1_n_est', 100, 500),
        learning_rate     = trial.suggest_float('s1_lr', 0.02, 0.2, log=True),
        num_leaves        = trial.suggest_int('s1_nl', 31, 255),
        min_child_samples = trial.suggest_int('s1_mcs', 10, 80),
        subsample         = trial.suggest_float('s1_ss', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('s1_cst', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('s1_ra', 1e-3, 3.0, log=True),
        reg_lambda        = trial.suggest_float('s1_rl', 1e-3, 3.0, log=True),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    # Stage 2 params (CATE models — проще, меньше деревьев)
    params2 = dict(
        n_estimators      = trial.suggest_int('s2_n_est', 50, 300),
        learning_rate     = trial.suggest_float('s2_lr', 0.02, 0.2, log=True),
        num_leaves        = trial.suggest_int('s2_nl', 15, 127),
        min_child_samples = trial.suggest_int('s2_mcs', 10, 80),
        subsample         = trial.suggest_float('s2_ss', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('s2_cst', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('s2_ra', 1e-3, 3.0, log=True),
        reg_lambda        = trial.suggest_float('s2_rl', 1e-3, 3.0, log=True),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    prop_model = trial.suggest_categorical('prop_model', ['logreg', 'lgb'])
    feat_set   = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
    feat_cols  = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP

    try:
        score, _ = xlearner_oof(params1, params2, prop_model,
                                folds_opt2, train_opt2, y_opt2, T_opt2, feat_cols)
        return score
    except Exception as e:
        return -999.0


study_xl = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)
)
# timeout=2700 = 45 минут на поиск, ~15 минут остаётся на финальный OOF
study_xl.optimize(objective_xl, n_trials=200, timeout=2700, show_progress_bar=True)

best_xl = study_xl.best_params
print(f'\nBest Optuna XL trial: uplift@10={study_xl.best_value:.4f}  '
      f'trials_done={len(study_xl.trials)}')
print(f'prop_model={best_xl["prop_model"]},  feat_set={best_xl["feat_set"]}')

# Масштабируем n_estimators для финального обучения
def scale_params(p, prefix, scale=2.5):
    return dict(
        n_estimators      = min(int(p[f'{prefix}_n_est'] * scale), 1500),
        learning_rate     = p[f'{prefix}_lr'],
        num_leaves        = p[f'{prefix}_nl'],
        min_child_samples = p[f'{prefix}_mcs'],
        subsample         = p[f'{prefix}_ss'],
        colsample_bytree  = p[f'{prefix}_cst'],
        reg_alpha         = p[f'{prefix}_ra'],
        reg_lambda        = p[f'{prefix}_rl'],
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )

best_params1_xl = scale_params(best_xl, 's1')
best_params2_xl = scale_params(best_xl, 's2')
best_prop_xl    = best_xl['prop_model']
best_feat_xl    = FEAT_COLS if best_xl['feat_set'] == 'all' else FEAT_COLS_TOP

# Финальный OOF на полных данных (5-fold)
print('\nRunning final 5-fold OOF on full train...')
_, oof_scores_xl_opt = xlearner_oof(
    best_params1_xl, best_params2_xl, best_prop_xl,
    folds, train, y, T, best_feat_xl
)

elapsed_xl = time.time() - t0
u10_xl = register_result('Optuna X-Learner', y, oof_scores_xl_opt, T, elapsed_xl)
np.save(f'{ARTIFACTS_DIR}/oof_xlearner_opt.npy', oof_scores_xl_opt)
with open(f'{ARTIFACTS_DIR}/optuna_xl_params.json', 'w') as f:
    json.dump({
        'params1': best_params1_xl,
        'params2': best_params2_xl,
        'prop_model': best_prop_xl,
        'feat_set': best_xl['feat_set'],
        'search_uplift10': study_xl.best_value,
        'trials_completed': len(study_xl.trials)
    }, f, indent=2)

In [ ]:
# ═══ MODEL 12: Per-Channel Optuna X-Learner ══════════════════════════════════
print('Training Model 12: Per-Channel Optuna X-Learner...')
t0 = time.time()

COMM_CODES = sorted(train[COMM_COL].unique())
print(f'Channels: {COMM_CODES}')
# Проверим размеры
for ch in COMM_CODES:
    mask = train[COMM_COL] == ch
    n_t  = T[mask].sum()
    n_c  = (1 - T[mask]).sum()
    print(f'  {ch}: n={mask.sum():,}  treated={n_t:,}  control={n_c:,}')

# ── Функция X-Learner OOF для произвольного подмножества ─────────────────────
def xlearner_oof_subset(params1, params2, prop_model, folds_list,
                        df_sub, y_sub, T_sub, feat_cols):
    """X-Learner OOF на произвольном подмножестве данных."""
    oof = np.zeros(len(df_sub))
    for tr_idx, val_idx in folds_list:
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]
        mask1 = Ttr == 1; mask0 = Ttr == 0

        mu1 = lgb.LGBMRegressor(**params1); mu0 = lgb.LGBMRegressor(**params1)
        mu1.fit(X_tr[mask1], ytr[mask1])
        mu0.fit(X_tr[mask0], ytr[mask0])

        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]

        tau1 = lgb.LGBMRegressor(**params2); tau0 = lgb.LGBMRegressor(**params2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        g.fit(X_tr, Ttr)
        g_val = g.predict_proba(X_val)[:, 1]

        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    return uplift_at_k_spend(y_sub, oof, T_sub, k=0.1), oof


# ── Per-channel Optuna + финальный OOF ───────────────────────────────────────
channel_params  = {}   # лучшие параметры по каналу
channel_studies = {}
oof_perchan_xl  = np.zeros(len(train))
test_scores_perchan_xl = np.zeros(len(test))

for ch in COMM_CODES:
    print(f'\n{"="*60}')
    print(f'  Channel: {ch}')
    t_ch = time.time()

    # Маски для трейна
    ch_mask_train = (train[COMM_COL] == ch).values
    df_ch   = train[ch_mask_train].reset_index(drop=True)
    y_ch    = y[ch_mask_train]
    T_ch    = T[ch_mask_train]

    # Маска для теста
    ch_mask_test = (test[COMM_COL] == ch).values
    df_ch_test   = test[ch_mask_test].reset_index(drop=True)

    n_ch = len(df_ch)
    print(f'  Train: {n_ch:,}  treated={T_ch.sum():,}  control={(1-T_ch).sum():,}')
    print(f'  Test:  {ch_mask_test.sum():,}')

    # ── 40% subsample для Optuna ─────────────────────────────────────────────
    sss_ch = StratifiedShuffleSplit(n_splits=1, test_size=0.6,
                                    random_state=RANDOM_STATE + hash(ch) % 100)
    opt_idx_ch, _ = next(sss_ch.split(df_ch, make_stratify_col(df_ch)))
    df_opt_ch  = df_ch.iloc[opt_idx_ch].reset_index(drop=True)
    y_opt_ch   = y_ch[opt_idx_ch]
    T_opt_ch   = T_ch[opt_idx_ch]
    folds_opt_ch = list(
        StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
        .split(df_opt_ch, make_stratify_col(df_opt_ch))
    )

    def make_objective_ch(df_o, y_o, T_o, f_opt_ch):
        """Замыкание: objective для конкретного канала."""
        def objective_ch(trial):
            params1 = dict(
                n_estimators      = trial.suggest_int('s1_n_est', 100, 500),
                learning_rate     = trial.suggest_float('s1_lr', 0.02, 0.2, log=True),
                num_leaves        = trial.suggest_int('s1_nl', 31, 255),
                min_child_samples = trial.suggest_int('s1_mcs', 10, 80),
                subsample         = trial.suggest_float('s1_ss', 0.6, 1.0),
                colsample_bytree  = trial.suggest_float('s1_cst', 0.5, 1.0),
                reg_alpha         = trial.suggest_float('s1_ra', 1e-3, 3.0, log=True),
                reg_lambda        = trial.suggest_float('s1_rl', 1e-3, 3.0, log=True),
                n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
            )
            params2 = dict(
                n_estimators      = trial.suggest_int('s2_n_est', 50, 300),
                learning_rate     = trial.suggest_float('s2_lr', 0.02, 0.2, log=True),
                num_leaves        = trial.suggest_int('s2_nl', 15, 127),
                min_child_samples = trial.suggest_int('s2_mcs', 10, 80),
                subsample         = trial.suggest_float('s2_ss', 0.6, 1.0),
                colsample_bytree  = trial.suggest_float('s2_cst', 0.5, 1.0),
                reg_alpha         = trial.suggest_float('s2_ra', 1e-3, 3.0, log=True),
                reg_lambda        = trial.suggest_float('s2_rl', 1e-3, 3.0, log=True),
                n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
            )
            prop_model = trial.suggest_categorical('prop_model', ['logreg', 'lgb'])
            feat_set   = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
            feat_cols  = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP
            try:
                score, _ = xlearner_oof_subset(
                    params1, params2, prop_model,
                    f_opt_ch, df_o, y_o, T_o, feat_cols
                )
                return score
            except Exception as e:
                return -999.0
        return objective_ch

    # ── Optuna для канала ────────────────────────────────────────────────────
    study_ch = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)
    )
    # ~15 мин на канал (3 канала = ~45 мин)
    study_ch.optimize(
        make_objective_ch(df_opt_ch, y_opt_ch, T_opt_ch, folds_opt_ch),
        n_trials=60, timeout=900, show_progress_bar=True
    )
    best_ch = study_ch.best_params
    channel_studies[ch] = study_ch
    print(f'  Best trial uplift@10={study_ch.best_value:.4f}  '
          f'trials={len(study_ch.trials)}')
    print(f'  prop={best_ch["prop_model"]}  feat={best_ch["feat_set"]}')

    # ── Масштабируем n_estimators для финального обучения ────────────────────
    def scale_ch(p, prefix, scale=2.5):
        return dict(
            n_estimators      = min(int(p[f'{prefix}_n_est'] * scale), 1500),
            learning_rate     = p[f'{prefix}_lr'],
            num_leaves        = p[f'{prefix}_nl'],
            min_child_samples = p[f'{prefix}_mcs'],
            subsample         = p[f'{prefix}_ss'],
            colsample_bytree  = p[f'{prefix}_cst'],
            reg_alpha         = p[f'{prefix}_ra'],
            reg_lambda        = p[f'{prefix}_rl'],
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
        )

    p1_ch   = scale_ch(best_ch, 's1')
    p2_ch   = scale_ch(best_ch, 's2')
    prop_ch = best_ch['prop_model']
    feat_ch = FEAT_COLS if best_ch['feat_set'] == 'all' else FEAT_COLS_TOP
    channel_params[ch] = {'params1': p1_ch, 'params2': p2_ch,
                          'prop': prop_ch, 'feat': best_ch['feat_set']}

    # ── Финальный OOF на канале (5-fold) ─────────────────────────────────────
    folds_ch_full = list(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        .split(df_ch, make_stratify_col(df_ch))
    )
    u10_ch, oof_ch = xlearner_oof_subset(
        p1_ch, p2_ch, prop_ch,
        folds_ch_full, df_ch, y_ch, T_ch, feat_ch
    )
    oof_perchan_xl[ch_mask_train] = oof_ch
    print(f'  Final OOF uplift@10={u10_ch:.4f}  ({time.time()-t_ch:.0f}s)')

    # ── Предсказания на тесте для канала ─────────────────────────────────────
    X_tr_full = encode_X(df_ch, feat_ch)
    X_te_ch   = encode_X(df_ch_test, feat_ch)
    mask1_f   = T_ch == 1; mask0_f = T_ch == 0

    mu1_f = lgb.LGBMRegressor(**p1_ch); mu0_f = lgb.LGBMRegressor(**p1_ch)
    mu1_f.fit(X_tr_full[mask1_f], y_ch[mask1_f])
    mu0_f.fit(X_tr_full[mask0_f], y_ch[mask0_f])

    D1_f = y_ch[mask1_f] - mu0_f.predict(X_tr_full[mask1_f])
    D0_f = mu1_f.predict(X_tr_full[mask0_f]) - y_ch[mask0_f]

    tau1_f = lgb.LGBMRegressor(**p2_ch); tau0_f = lgb.LGBMRegressor(**p2_ch)
    tau1_f.fit(X_tr_full[mask1_f], D1_f)
    tau0_f.fit(X_tr_full[mask0_f], D0_f)

    if prop_ch == 'logreg':
        g_f = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
    else:
        g_f = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                  n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    g_f.fit(X_tr_full, T_ch)
    g_te = g_f.predict_proba(X_te_ch)[:, 1]

    test_scores_perchan_xl[ch_mask_test] = (
        g_te * tau0_f.predict(X_te_ch) + (1 - g_te) * tau1_f.predict(X_te_ch)
    )

# ── Итоговые метрики ──────────────────────────────────────────────────────────
elapsed_pcxl = time.time() - t0
u10_pcxl = uplift_at_k_spend(y, oof_perchan_xl, T, k=0.1)
u20_pcxl = uplift_at_k_spend(y, oof_perchan_xl, T, k=0.2)
print(f'\n{"="*60}')
print(f'Per-Channel Optuna X-Learner:')
print(f'  uplift@10={u10_pcxl:.4f}  uplift@20={u20_pcxl:.4f}')
print(f'  elapsed={elapsed_pcxl:.0f}s')

# Per-channel breakdown
print('\nPer-channel OOF uplift@10:')
for ch in COMM_CODES:
    mask = (train[COMM_COL] == ch).values
    u_ch = uplift_at_k_spend(y[mask], oof_perchan_xl[mask], T[mask], k=0.1)
    print(f'  {ch}: {u_ch:.4f}')

np.save(f'{ARTIFACTS_DIR}/oof_perchan_xl_opt.npy', oof_perchan_xl)

# ── Сабмит ────────────────────────────────────────────────────────────────────
submission_pcxl = pd.DataFrame({
    'user_id':      test['user_id'],
    'UPLIFT_SCORE': test_scores_perchan_xl
})
sub_path_pcxl = f'{ARTIFACTS_DIR}/submission_perchan_xl_opt.csv'
submission_pcxl.to_csv(sub_path_pcxl, index=False)
print(f'\n✅ Saved: {sub_path_pcxl}')
print(f'   score stats: min={test_scores_perchan_xl.min():.3f}  '
      f'max={test_scores_perchan_xl.max():.3f}  '
      f'mean={test_scores_perchan_xl.mean():.3f}')

# Сохраняем параметры по каналам
with open(f'{ARTIFACTS_DIR}/perchan_xl_params.json', 'w') as f:
    json.dump({str(ch): {k: v for k, v in d.items() if k != 'feat'}
               for ch, d in channel_params.items()}, f, indent=2)

# ── Бленд с глобальной моделью ────────────────────────────────────────────────
print('\n--- Blend: per-channel XL vs global XL ---')
oof_global = np.load(f'{ARTIFACTS_DIR}/oof_xlearner_opt.npy')
from scipy.stats import rankdata

for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    r1 = rankdata(oof_perchan_xl)
    r2 = rankdata(oof_global)
    blend = w * r1 + (1 - w) * r2
    u10_b = uplift_at_k_spend(y, blend, T, k=0.1)
    print(f'  perchan*{w:.1f} + global*{1-w:.1f}: uplift@10={u10_b:.4f}')

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata

# ── Загружаем тестовые скоры ──────────────────────────────────────────────────
sub_perchan = pd.read_csv(f'{ARTIFACTS_DIR}/submission_perchan_xl_opt.csv')
sub_global  = pd.read_csv(f'{ARTIFACTS_DIR}/submission_xl_opt.csv')  # имя глобального сабмита

# Проверяем что user_id совпадает
assert (sub_perchan['user_id'].values == sub_global['user_id'].values).all(), \
    'user_id не совпадают!'

scores_perchan = sub_perchan['UPLIFT_SCORE'].values
scores_global  = sub_global['UPLIFT_SCORE'].values

# ── Бленд 0.7 perchan + 0.3 global по рангу ──────────────────────────────────
# Важно: rankdata применяем к тесту, а не к трейну
blend_test = 0.7 * rankdata(scores_perchan) + 0.3 * rankdata(scores_global)

submission_blend = pd.DataFrame({
    'user_id':      sub_perchan['user_id'],
    'UPLIFT_SCORE': blend_test
})

sub_path_blend = f'{ARTIFACTS_DIR}/submission_blend_perchan07_global03.csv'
submission_blend.to_csv(sub_path_blend, index=False)
print(f'✅ Saved: {sub_path_blend}')
print(f'   n={len(submission_blend):,}')
print(f'   score stats: min={blend_test.min():.1f}  max={blend_test.max():.1f}  mean={blend_test.mean():.1f}')

# Быстрая проверка: распределение по каналам в топ-10% теста
n_top = int(0.1 * len(test))
top_mask = blend_test >= np.sort(blend_test)[::-1][n_top]
print(f'\nСостав топ-10% теста ({n_top:,} клиентов):')
for ch in sorted(test[COMM_COL].unique()):
    n_ch = ((test[COMM_COL] == ch) & top_mask).sum()
    print(f'  ch={ch}: {n_ch:,} ({n_ch/n_top*100:.1f}%)')